In [1]:
import duckdb
import os

# Connexion DuckDB
con = duckdb.connect()

# Installation et activation du module S3
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Connexion à RustFS
con.execute(f"""
    CREATE OR REPLACE SECRET rustfs_secret (
        TYPE S3,
        KEY_ID '{os.getenv("S3_ACCESS_KEY")}',
        SECRET '{os.getenv("S3_SECRET_KEY")}',
        ENDPOINT 'rustfs:9000',
        URL_STYLE 'path',
        USE_SSL false
    )
""")

print("DuckDB est connecté à RustFS !")

DuckDB est connecté à RustFS !


In [2]:
# Vérifier le nombre de lignes dans la table orders d'Instacart
resultat = con.execute("""
    SELECT COUNT(*)
    FROM read_parquet('s3://bronze/instacart/raw/orders.parquet')
""").fetchone()

print(f"Nombre de commandes : {resultat[0]:,}")

Nombre de commandes : 3,421,083


In [3]:
# Afficher la structure de la table orders
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('s3://bronze/instacart/raw/orders.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,user_id,VARCHAR,YES,None,None,None
2,eval_set,VARCHAR,YES,None,None,None
3,order_number,VARCHAR,YES,None,None,None
4,order_dow,VARCHAR,YES,None,None,None
5,order_hour_of_day,VARCHAR,YES,None,None,None
6,days_since_prior_order,VARCHAR,YES,None,None,None


In [4]:
# Afficher les 10 premières lignes
con.execute("""
    SELECT *
    FROM read_parquet('s3://bronze/instacart/raw/orders.parquet')
    LIMIT 10
""").df()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,08,
1,2398795,1,prior,2,3,07,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,07,29.0
4,431534,1,prior,5,4,15,28.0
5,3367565,1,prior,6,2,07,19.0
6,550135,1,prior,7,1,09,20.0
7,3108588,1,prior,8,1,14,14.0
8,2295261,1,prior,9,1,16,0.0
9,2550362,1,prior,10,4,08,30.0


In [5]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(*) FILTER (WHERE TRY_CAST(order_id AS BIGINT) IS NULL) AS order_id_invalides,
        COUNT(*) FILTER (WHERE TRY_CAST(user_id AS BIGINT) IS NULL) AS user_id_invalides,
        COUNT(*) FILTER (WHERE TRY_CAST(order_number AS INTEGER) IS NULL) AS order_number_invalides,
        COUNT(*) FILTER (WHERE TRY_CAST(order_dow AS INTEGER) NOT BETWEEN 0 AND 6) AS jours_invalides,
        COUNT(*) FILTER (WHERE TRY_CAST(order_hour_of_day AS INTEGER) NOT BETWEEN 0 AND 23) AS heures_invalides,
        COUNT(*) FILTER (WHERE NULLIF(TRIM(days_since_prior_order), '') IS NULL) AS delais_manquants,
        COUNT(DISTINCT eval_set) AS nombre_types_commandes
    FROM read_parquet('s3://bronze/instacart/raw/orders.parquet')
""").df()

,total_lignes,order_id_invalides,user_id_invalides,order_number_invalides,jours_invalides,heures_invalides,delais_manquants,nombre_types_commandes
0,3421083,0,0,0,0,0,206209,3


In [6]:
resultat = con.execute("""
    SELECT
        COUNT(*) AS total,
        COUNT(DISTINCT order_id) AS commandes_distinctes,
        COUNT(DISTINCT user_id) AS utilisateurs,
        COUNT(DISTINCT eval_set) AS types_commandes,
        COUNT(*) FILTER (
            WHERE TRY_CAST(order_id AS BIGINT) IS NULL
        ) AS order_id_invalides
    FROM read_parquet('s3://bronze/instacart/raw/orders.parquet')
""").fetchone()

for nom, valeur in zip(
    ["Total", "Commandes distinctes", "Utilisateurs",
     "Types de commandes", "Order_id invalides"],
    resultat
):
    print(f"{nom} : {valeur}")

Total : 3421083
Commandes distinctes : 3421083
Utilisateurs : 206209
Types de commandes : 3
Order_id invalides : 0


In [7]:
con.execute("""
    CREATE OR REPLACE TABLE orders_silver AS

    SELECT
        TRY_CAST(order_id AS BIGINT) AS order_id,
        TRY_CAST(user_id AS BIGINT) AS user_id,
        LOWER(TRIM(eval_set)) AS eval_set,
        TRY_CAST(order_number AS INTEGER) AS order_number,
        TRY_CAST(order_dow AS INTEGER) AS order_dow,
        TRY_CAST(order_hour_of_day AS INTEGER) AS order_hour_of_day,
        TRY_CAST(NULLIF(TRIM(days_since_prior_order), '') AS DOUBLE)
            AS days_since_prior_order

    FROM read_parquet('s3://bronze/instacart/raw/orders.parquet')

    WHERE
        TRY_CAST(order_id AS BIGINT) IS NOT NULL
        AND TRY_CAST(user_id AS BIGINT) IS NOT NULL
        AND TRY_CAST(order_number AS INTEGER) > 0
        AND TRY_CAST(order_dow AS INTEGER) BETWEEN 0 AND 6
        AND TRY_CAST(order_hour_of_day AS INTEGER) BETWEEN 0 AND 23
        AND LOWER(TRIM(eval_set)) IN ('prior', 'train', 'test')
""")

print("Table orders_silver créée !")

Table orders_silver créée !


In [8]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT order_id) AS commandes_distinctes,
        COUNT(DISTINCT user_id) AS utilisateurs,
        COUNT(*) FILTER (WHERE days_since_prior_order IS NULL) AS delais_manquants,
        COUNT(*) FILTER (WHERE order_dow NOT BETWEEN 0 AND 6) AS jours_invalides,
        COUNT(*) FILTER (WHERE order_hour_of_day NOT BETWEEN 0 AND 23) AS heures_invalides
    FROM orders_silver
""").df()

,total_lignes,commandes_distinctes,utilisateurs,delais_manquants,jours_invalides,heures_invalides
0,3421083,3421083,206209,206209,0,0


In [9]:
%pip install boto3

Note: you may need to restart the kernel to use updated packages.


In [1]:
import boto3
import os

s3 = boto3.client(
    "s3",
    endpoint_url="http://rustfs:9000",
    aws_access_key_id=os.getenv("S3_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("S3_SECRET_KEY"),
    region_name="us-east-1"
)

print("Connexion à RustFS réussie !")

Connexion à RustFS réussie !


In [2]:
bucket_name = "silver"

buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]

if bucket_name not in buckets:
    s3.create_bucket(Bucket=bucket_name)
    print("Bucket silver créé avec succès !")
else:
    print("Le bucket silver existe déjà !")

Bucket silver créé avec succès !


In [4]:
import duckdb
import os

con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

con.execute(f"""
    CREATE OR REPLACE SECRET rustfs_secret (
        TYPE S3,
        KEY_ID '{os.getenv("S3_ACCESS_KEY")}',
        SECRET '{os.getenv("S3_SECRET_KEY")}',
        ENDPOINT 'rustfs:9000',
        URL_STYLE 'path',
        USE_SSL false
    )
""")

print("Connexion DuckDB rétablie !")

Connexion DuckDB rétablie !


In [5]:
con.execute("""
    CREATE OR REPLACE TABLE orders_silver AS
    SELECT
        TRY_CAST(order_id AS BIGINT) AS order_id,
        TRY_CAST(user_id AS BIGINT) AS user_id,
        LOWER(TRIM(eval_set)) AS eval_set,
        TRY_CAST(order_number AS INTEGER) AS order_number,
        TRY_CAST(order_dow AS INTEGER) AS order_dow,
        TRY_CAST(order_hour_of_day AS INTEGER) AS order_hour_of_day,
        TRY_CAST(NULLIF(TRIM(days_since_prior_order), '') AS DOUBLE)
            AS days_since_prior_order
    FROM read_parquet('s3://bronze/instacart/raw/orders.parquet')
    WHERE
        TRY_CAST(order_id AS BIGINT) IS NOT NULL
        AND TRY_CAST(user_id AS BIGINT) IS NOT NULL
        AND TRY_CAST(order_number AS INTEGER) > 0
        AND TRY_CAST(order_dow AS INTEGER) BETWEEN 0 AND 6
        AND TRY_CAST(order_hour_of_day AS INTEGER) BETWEEN 0 AND 23
        AND LOWER(TRIM(eval_set)) IN ('prior', 'train', 'test')
""")

print("Table orders_silver recréée !")

Table orders_silver recréée !


In [6]:
con.execute("""
    COPY orders_silver
    TO 's3://silver/instacart/orders.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Fichier orders.parquet exporté dans Silver !")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fichier orders.parquet exporté dans Silver !


In [7]:
response = s3.list_objects_v2(
    Bucket="silver",
    Prefix="instacart/"
)

for obj in response.get("Contents", []):
    print(f"{obj['Key']} - {obj['Size'] / (1024 * 1024):.2f} Mo")

instacart/orders.parquet - 18.60 Mo


In [8]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('s3://bronze/instacart/raw/products.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,product_id,VARCHAR,YES,None,None,None
1,product_name,VARCHAR,YES,None,None,None
2,aisle_id,VARCHAR,YES,None,None,None
3,department_id,VARCHAR,YES,None,None,None


In [9]:
con.execute("""
    SELECT
        COUNT(*) AS total_produits,
        COUNT(DISTINCT product_id) AS produits_distincts,
        COUNT(*) FILTER (
            WHERE TRY_CAST(product_id AS BIGINT) IS NULL
        ) AS product_id_invalides,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(product_name), '') IS NULL
        ) AS noms_manquants,
        COUNT(*) FILTER (
            WHERE TRY_CAST(aisle_id AS INTEGER) IS NULL
        ) AS aisle_id_invalides,
        COUNT(*) FILTER (
            WHERE TRY_CAST(department_id AS INTEGER) IS NULL
        ) AS department_id_invalides
    FROM read_parquet('s3://bronze/instacart/raw/products.parquet')
""").df()

,total_produits,produits_distincts,product_id_invalides,noms_manquants,aisle_id_invalides,department_id_invalides
0,49688,49688,0,0,0,0


In [10]:
con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE a.aisle_id IS NULL
        ) AS rayons_introuvables,

        COUNT(*) FILTER (
            WHERE d.department_id IS NULL
        ) AS departements_introuvables

    FROM read_parquet('s3://bronze/instacart/raw/products.parquet') p

    LEFT JOIN read_parquet('s3://bronze/instacart/raw/aisles.parquet') a
        ON TRY_CAST(p.aisle_id AS INTEGER) = TRY_CAST(a.aisle_id AS INTEGER)

    LEFT JOIN read_parquet('s3://bronze/instacart/raw/departments.parquet') d
        ON TRY_CAST(p.department_id AS INTEGER) = TRY_CAST(d.department_id AS INTEGER)
""").df()

,rayons_introuvables,departements_introuvables
0,0,0


In [11]:
con.execute("""
    CREATE OR REPLACE TABLE products_silver AS

    SELECT
        TRY_CAST(product_id AS BIGINT) AS product_id,
        TRIM(product_name) AS product_name,
        TRY_CAST(aisle_id AS BIGINT) AS aisle_id,
        TRY_CAST(department_id AS BIGINT) AS department_id

    FROM read_parquet('s3://bronze/instacart/raw/products.parquet')

    WHERE
        TRY_CAST(product_id AS BIGINT) IS NOT NULL
        AND NULLIF(TRIM(product_name), '') IS NOT NULL
        AND TRY_CAST(aisle_id AS BIGINT) IS NOT NULL
        AND TRY_CAST(department_id AS BIGINT) IS NOT NULL

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY TRY_CAST(product_id AS BIGINT)
        ORDER BY product_name
    ) = 1
""")

print("Table products_silver créée !")

Table products_silver créée !


In [12]:
con.execute("""
    SELECT
        COUNT(*) AS total_produits,
        COUNT(DISTINCT product_id) AS produits_distincts,
        COUNT(*) FILTER (WHERE product_name IS NULL OR TRIM(product_name) = '') AS noms_invalides,
        COUNT(*) FILTER (WHERE aisle_id IS NULL) AS rayons_manquants,
        COUNT(*) FILTER (WHERE department_id IS NULL) AS departements_manquants
    FROM products_silver
""").df()

,total_produits,produits_distincts,noms_invalides,rayons_manquants,departements_manquants
0,49688,49688,0,0,0


In [13]:
con.execute("DESCRIBE products_silver").df()

,column_name,column_type,null,key,default,extra
0,product_id,BIGINT,YES,None,None,None
1,product_name,VARCHAR,YES,None,None,None
2,aisle_id,BIGINT,YES,None,None,None
3,department_id,BIGINT,YES,None,None,None


In [14]:
con.execute("""
    COPY products_silver
    TO 's3://silver/instacart/products.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Fichier products.parquet exporté dans Silver !")

Fichier products.parquet exporté dans Silver !


In [15]:
response = s3.list_objects_v2(
    Bucket="silver",
    Prefix="instacart/"
)

for obj in response.get("Contents", []):
    print(f"{obj['Key']} - {obj['Size'] / (1024 * 1024):.2f} Mo")

instacart/orders.parquet - 18.60 Mo
instacart/products.parquet - 0.70 Mo


In [16]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('s3://bronze/instacart/raw/aisles.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,aisle_id,VARCHAR,YES,None,None,None
1,aisle,VARCHAR,YES,None,None,None


In [17]:
con.execute("""
    SELECT
        COUNT(*) AS total_rayons,
        COUNT(DISTINCT aisle_id) AS rayons_distincts,
        COUNT(*) FILTER (
            WHERE TRY_CAST(aisle_id AS INTEGER) IS NULL
        ) AS identifiants_invalides,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(aisle), '') IS NULL
        ) AS noms_manquants
    FROM read_parquet('s3://bronze/instacart/raw/aisles.parquet')
""").df()

,total_rayons,rayons_distincts,identifiants_invalides,noms_manquants
0,134,134,0,0


In [18]:
con.execute("""
    SELECT aisle_id
    FROM read_parquet('s3://bronze/instacart/raw/aisles.parquet')
    LIMIT 15
""").df()

,aisle_id
0,1
1,2
2,3
3,4
4,5
5,6
6,7
7,8
8,9
9,10


In [19]:
con.execute("""
    CREATE OR REPLACE TABLE aisles_silver AS

    SELECT
        TRY_CAST(aisle_id AS INTEGER) AS aisle_id,
        TRIM(aisle) AS aisle

    FROM read_parquet('s3://bronze/instacart/raw/aisles.parquet')

    WHERE
        TRY_CAST(aisle_id AS INTEGER) IS NOT NULL
        AND NULLIF(TRIM(aisle), '') IS NOT NULL

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY TRY_CAST(aisle_id AS INTEGER)
        ORDER BY aisle
    ) = 1
""")

print("Table aisles_silver créée !")

Table aisles_silver créée !


In [20]:
con.execute("""
    SELECT
        COUNT(*) AS total_rayons,
        COUNT(DISTINCT aisle_id) AS rayons_distincts,
        COUNT(*) FILTER (
            WHERE aisle IS NULL OR TRIM(aisle) = ''
        ) AS noms_invalides
    FROM aisles_silver
""").df()

,total_rayons,rayons_distincts,noms_invalides
0,134,134,0


In [21]:
con.execute("""
    COPY aisles_silver
    TO 's3://silver/instacart/aisles.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Fichier aisles.parquet exporté dans Silver !")

Fichier aisles.parquet exporté dans Silver !


In [22]:
response = s3.list_objects_v2(
    Bucket="silver",
    Prefix="instacart/"
)

for obj in response.get("Contents", []):
    print(f"{obj['Key']} - {obj['Size'] / (1024 * 1024):.2f} Mo")

instacart/aisles.parquet - 0.00 Mo
instacart/orders.parquet - 18.60 Mo
instacart/products.parquet - 0.70 Mo


In [23]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('s3://bronze/instacart/raw/departments.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,department_id,VARCHAR,YES,None,None,None
1,department,VARCHAR,YES,None,None,None


In [24]:
con.execute("""
    SELECT
        COUNT(*) AS total_departements,
        COUNT(DISTINCT department_id) AS departements_distincts,
        COUNT(*) FILTER (
            WHERE TRY_CAST(department_id AS INTEGER) IS NULL
        ) AS identifiants_invalides,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(department), '') IS NULL
        ) AS noms_manquants
    FROM read_parquet('s3://bronze/instacart/raw/departments.parquet')
""").df()

,total_departements,departements_distincts,identifiants_invalides,noms_manquants
0,21,21,0,0


In [25]:
con.execute("""
    CREATE OR REPLACE TABLE departments_silver AS

    SELECT
        TRY_CAST(department_id AS INTEGER) AS department_id,
        TRIM(department) AS department

    FROM read_parquet('s3://bronze/instacart/raw/departments.parquet')

    WHERE
        TRY_CAST(department_id AS INTEGER) IS NOT NULL
        AND NULLIF(TRIM(department), '') IS NOT NULL

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY TRY_CAST(department_id AS INTEGER)
        ORDER BY department
    ) = 1
""")

print("Table departments_silver créée !")

Table departments_silver créée !


In [26]:
con.execute("""
    SELECT
        COUNT(*) AS total_departements,
        COUNT(DISTINCT department_id) AS departements_distincts,
        COUNT(*) FILTER (
            WHERE department IS NULL OR TRIM(department) = ''
        ) AS noms_invalides
    FROM departments_silver
""").df()

,total_departements,departements_distincts,noms_invalides
0,21,21,0


In [27]:
con.execute("""
    COPY departments_silver
    TO 's3://silver/instacart/departments.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Fichier departments.parquet exporté dans Silver !")

Fichier departments.parquet exporté dans Silver !


In [28]:
response = s3.list_objects_v2(
    Bucket="silver",
    Prefix="instacart/"
)

for obj in response.get("Contents", []):
    print(f"{obj['Key']} - {obj['Size'] / (1024 * 1024):.2f} Mo")

instacart/aisles.parquet - 0.00 Mo
instacart/departments.parquet - 0.00 Mo
instacart/orders.parquet - 18.60 Mo
instacart/products.parquet - 0.70 Mo


In [29]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('s3://bronze/instacart/raw/order_products__prior.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,product_id,VARCHAR,YES,None,None,None
2,add_to_cart_order,VARCHAR,YES,None,None,None
3,reordered,VARCHAR,YES,None,None,None


In [30]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,

        COUNT(*) FILTER (
            WHERE TRY_CAST(order_id AS BIGINT) IS NULL
        ) AS order_id_invalides,

        COUNT(*) FILTER (
            WHERE TRY_CAST(product_id AS BIGINT) IS NULL
        ) AS product_id_invalides,

        COUNT(*) FILTER (
            WHERE TRY_CAST(add_to_cart_order AS INTEGER) IS NULL
               OR TRY_CAST(add_to_cart_order AS INTEGER) <= 0
        ) AS position_panier_invalides,

        COUNT(*) FILTER (
            WHERE TRY_CAST(reordered AS INTEGER) NOT IN (0, 1)
        ) AS reordered_invalides,

        COUNT(DISTINCT order_id) AS commandes_distinctes,
        COUNT(DISTINCT product_id) AS produits_distincts

    FROM read_parquet('s3://bronze/instacart/raw/order_products__prior.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_lignes,order_id_invalides,product_id_invalides,position_panier_invalides,reordered_invalides,commandes_distinctes,produits_distincts
0,32434489,0,0,0,0,3214874,49677


In [31]:
con.execute("""
    SELECT
        COUNT(*) AS associations_invalides
    FROM read_parquet('s3://bronze/instacart/raw/order_products__prior.parquet') op
    LEFT JOIN read_parquet('s3://bronze/instacart/raw/orders.parquet') o
        ON TRY_CAST(op.order_id AS BIGINT) = TRY_CAST(o.order_id AS BIGINT)
    LEFT JOIN read_parquet('s3://bronze/instacart/raw/products.parquet') p
        ON TRY_CAST(op.product_id AS BIGINT) = TRY_CAST(p.product_id AS BIGINT)
    WHERE o.order_id IS NULL
       OR p.product_id IS NULL
       OR o.eval_set <> 'prior'
""").df()

,associations_invalides
0,0


In [32]:
con.execute("""
    SELECT COUNT(*) AS couples_en_doublon
    FROM (
        SELECT
            order_id,
            product_id
        FROM read_parquet('s3://bronze/instacart/raw/order_products__prior.parquet')
        GROUP BY order_id, product_id
        HAVING COUNT(*) > 1
    ) d
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,couples_en_doublon
0,0


In [33]:
con.execute("""
    CREATE OR REPLACE TABLE order_products_prior_silver AS
    SELECT
        TRY_CAST(op.order_id AS BIGINT) AS order_id,
        TRY_CAST(op.product_id AS BIGINT) AS product_id,
        TRY_CAST(op.add_to_cart_order AS INTEGER) AS add_to_cart_order,
        TRY_CAST(op.reordered AS INTEGER) AS reordered
    FROM read_parquet('s3://bronze/instacart/raw/order_products__prior.parquet') op
    INNER JOIN orders_silver o
        ON TRY_CAST(op.order_id AS BIGINT) = o.order_id
       AND o.eval_set = 'prior'
    INNER JOIN products_silver p
        ON TRY_CAST(op.product_id AS BIGINT) = p.product_id
    WHERE
        TRY_CAST(op.add_to_cart_order AS INTEGER) > 0
        AND TRY_CAST(op.reordered AS INTEGER) IN (0, 1)
""")

print("Table order_products_prior_silver créée !")

Table order_products_prior_silver créée !


In [34]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT order_id) AS commandes_distinctes,
        COUNT(DISTINCT product_id) AS produits_distincts,
        COUNT(*) FILTER (WHERE reordered NOT IN (0, 1)) AS reordered_invalides
    FROM order_products_prior_silver
""").df()

,total_lignes,commandes_distinctes,produits_distincts,reordered_invalides
0,32434489,3214874,49677,0


In [35]:
con.execute("""
    COPY order_products_prior_silver
    TO 's3://silver/instacart/order_products_prior.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Fichier order_products_prior.parquet exporté dans Silver !")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Fichier order_products_prior.parquet exporté dans Silver !


In [36]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('s3://bronze/instacart/raw/order_products__train.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,product_id,VARCHAR,YES,None,None,None
2,add_to_cart_order,VARCHAR,YES,None,None,None
3,reordered,VARCHAR,YES,None,None,None


In [37]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(*) FILTER (
            WHERE TRY_CAST(order_id AS BIGINT) IS NULL
        ) AS order_id_invalides,
        COUNT(*) FILTER (
            WHERE TRY_CAST(product_id AS BIGINT) IS NULL
        ) AS product_id_invalides,
        COUNT(*) FILTER (
            WHERE TRY_CAST(add_to_cart_order AS INTEGER) IS NULL
               OR TRY_CAST(add_to_cart_order AS INTEGER) <= 0
        ) AS position_panier_invalides,
        COUNT(*) FILTER (
            WHERE TRY_CAST(reordered AS INTEGER) NOT IN (0, 1)
        ) AS reordered_invalides,
        COUNT(DISTINCT order_id) AS commandes_distinctes,
        COUNT(DISTINCT product_id) AS produits_distincts
    FROM read_parquet('s3://bronze/instacart/raw/order_products__train.parquet')
""").df()

,total_lignes,order_id_invalides,product_id_invalides,position_panier_invalides,reordered_invalides,commandes_distinctes,produits_distincts
0,1384617,0,0,0,0,131209,39123


In [38]:
con.execute("""
    SELECT
        COUNT(*) AS associations_invalides
    FROM read_parquet('s3://bronze/instacart/raw/order_products__train.parquet') op
    LEFT JOIN orders_silver o
        ON TRY_CAST(op.order_id AS BIGINT) = o.order_id
    LEFT JOIN products_silver p
        ON TRY_CAST(op.product_id AS BIGINT) = p.product_id
    WHERE o.order_id IS NULL
       OR p.product_id IS NULL
       OR o.eval_set <> 'train'
""").df()

,associations_invalides
0,0


In [39]:
con.execute("""
    SELECT COUNT(*) AS couples_en_doublon
    FROM (
        SELECT order_id, product_id
        FROM read_parquet('s3://bronze/instacart/raw/order_products__train.parquet')
        GROUP BY order_id, product_id
        HAVING COUNT(*) > 1
    ) d
""").df()

,couples_en_doublon
0,0


In [40]:
con.execute("""
    CREATE OR REPLACE TABLE order_products_train_silver AS
    SELECT
        TRY_CAST(op.order_id AS BIGINT) AS order_id,
        TRY_CAST(op.product_id AS BIGINT) AS product_id,
        TRY_CAST(op.add_to_cart_order AS INTEGER) AS add_to_cart_order,
        TRY_CAST(op.reordered AS INTEGER) AS reordered
    FROM read_parquet('s3://bronze/instacart/raw/order_products__train.parquet') op
    INNER JOIN orders_silver o
        ON TRY_CAST(op.order_id AS BIGINT) = o.order_id
       AND o.eval_set = 'train'
    INNER JOIN products_silver p
        ON TRY_CAST(op.product_id AS BIGINT) = p.product_id
    WHERE
        TRY_CAST(op.add_to_cart_order AS INTEGER) > 0
        AND TRY_CAST(op.reordered AS INTEGER) IN (0, 1)
""")

print("Table order_products_train_silver créée !")

Table order_products_train_silver créée !


In [41]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT order_id) AS commandes_distinctes,
        COUNT(DISTINCT product_id) AS produits_distincts,
        COUNT(*) FILTER (WHERE reordered NOT IN (0, 1)) AS reordered_invalides
    FROM order_products_train_silver
""").df()

,total_lignes,commandes_distinctes,produits_distincts,reordered_invalides
0,1384617,131209,39123,0


In [42]:
con.execute("""
    COPY order_products_train_silver
    TO 's3://silver/instacart/order_products_train.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Fichier order_products_train.parquet exporté dans Silver !")

Fichier order_products_train.parquet exporté dans Silver !


In [43]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,allergens_tags,VARCHAR,YES,None,None,None
1,brands,VARCHAR,YES,None,None,None
2,categories_tags,VARCHAR,YES,None,None,None
3,code,VARCHAR,YES,None,None,None
4,ingredients_text,VARCHAR,YES,None,None,None
5,nova_group,VARCHAR,YES,None,None,None
6,nutriments,VARCHAR,YES,None,None,None
7,nutriments_estimated,VARCHAR,YES,None,None,None
8,nutriscore_grade,VARCHAR,YES,None,None,None
9,nutrition_data,VARCHAR,YES,None,None,None


In [44]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT code) AS codes_distincts,
        COUNT(*) FILTER (
            WHERE code IS NULL OR TRIM(code) = ''
        ) AS codes_manquants,
        COUNT(*) FILTER (
            WHERE product_name IS NULL OR TRIM(product_name) = ''
        ) AS noms_manquants,
        COUNT(*) FILTER (
            WHERE LOWER(TRIM(nutriscore_grade))
                NOT IN ('a', 'b', 'c', 'd', 'e')
              AND NULLIF(TRIM(nutriscore_grade), '') IS NOT NULL
        ) AS nutriscores_invalides,
        COUNT(*) FILTER (
            WHERE TRY_CAST(nova_group AS INTEGER) NOT BETWEEN 1 AND 4
              AND NULLIF(TRIM(nova_group), '') IS NOT NULL
        ) AS nova_invalides,
        COUNT(*) FILTER (
            WHERE TRY_CAST(serving_quantity AS DOUBLE) < 0
        ) AS quantites_negatives
    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
""").df()

,total_lignes,codes_distincts,codes_manquants,noms_manquants,nutriscores_invalides,nova_invalides,quantites_negatives
0,100,100,0,0,1,0,0


In [45]:
con.execute("""
    SELECT
        TRIM(nova_group) AS nova_group,
        COUNT(*) AS nombre
    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
    GROUP BY TRIM(nova_group)
    ORDER BY nova_group
""").df()

,nova_group,nombre
0,1,11
1,2,1
2,3,15
3,4,72
4,None,1


In [46]:
con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE nutriments IS NULL OR TRIM(nutriments) = ''
        ) AS nutriments_manquants,
        COUNT(*) FILTER (
            WHERE ingredients_text IS NULL OR TRIM(ingredients_text) = ''
        ) AS ingredients_manquants,
        COUNT(*) FILTER (
            WHERE brands IS NULL OR TRIM(brands) = ''
        ) AS marques_manquantes,
        COUNT(*) FILTER (
            WHERE categories_tags IS NULL OR TRIM(categories_tags) = ''
        ) AS categories_manquantes,
        COUNT(*) FILTER (
            WHERE allergens_tags IS NULL OR TRIM(allergens_tags) = ''
        ) AS allergenes_manquants,
        COUNT(*) FILTER (
            WHERE nutriscore_grade IS NULL OR TRIM(nutriscore_grade) = ''
        ) AS nutriscores_manquants
    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
""").df()

,nutriments_manquants,ingredients_manquants,marques_manquantes,categories_manquantes,allergenes_manquants,nutriscores_manquants
0,0,0,0,0,0,0


In [47]:
con.execute("""
    SELECT
        code,
        product_name,
        nova_group,
        nutriscore_grade,
        serving_quantity,
        nutriments,
        nutrition_data
    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
    LIMIT 5
""").df()

,code,product_name,nova_group,nutriscore_grade,serving_quantity,nutriments,nutrition_data
0,3274080005003,isabelle,1,a,None,"{""added-sugars"": 0, ""added-sugars_100g"": 0, ""a...",on
1,3168930010265,cruesly mélange de noix,4,b,None,"{""added-sugars"": 14.13, ""added-sugars_100g"": 1...",on
2,3046920022651,Noir Intense,3,e,None,"{""added-sugars"": 30.55, ""added-sugars_100g"": 3...",on
3,3760020507350,Pur beurre de cacahuète,1,b,None,"{""added-sugars"": 0, ""added-sugars_100g"": 0, ""a...",on
4,3228857002344,Pain de mie american sandwich complet sans suc...,4,b,None,"{""added-sugars"": 0, ""added-sugars_100g"": 0, ""a...",on


In [48]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(*) FILTER (
            WHERE json_valid(nutriments)
        ) AS json_valides,
        COUNT(*) FILTER (
            WHERE NOT json_valid(nutriments)
        ) AS json_invalides,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(nutriments), '') IS NULL
        ) AS nutriments_vides
    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
""").df()

,total_lignes,json_valides,json_invalides,nutriments_vides
0,100,100,0,0


In [49]:
con.execute("""
    SELECT
        json_keys(nutriments) AS cles_nutritionnelles
    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
    LIMIT 3
""").df()

,cles_nutritionnelles
0,"[added-sugars, added-sugars_100g, added-sugars..."
1,"[added-sugars, added-sugars_100g, added-sugars..."
2,"[added-sugars, added-sugars_100g, added-sugars..."


In [50]:
con.execute("""
    CREATE OR REPLACE TABLE produits_silver AS
    SELECT
        NULLIF(TRIM(code), '') AS code,
        NULLIF(TRIM(product_name), '') AS product_name,
        NULLIF(TRIM(brands), '') AS brands,
        NULLIF(TRIM(categories_tags), '') AS categories_tags,
        NULLIF(TRIM(allergens_tags), '') AS allergens_tags,
        NULLIF(TRIM(ingredients_text), '') AS ingredients_text,

        CASE
            WHEN TRY_CAST(nova_group AS INTEGER) BETWEEN 1 AND 4
            THEN TRY_CAST(nova_group AS INTEGER)
            ELSE NULL
        END AS nova_group,

        LOWER(NULLIF(TRIM(nutriscore_grade), '')) AS nutriscore_grade,

        TRY_CAST(NULLIF(TRIM(serving_quantity), '') AS DOUBLE)
            AS serving_quantity,

        nutriments,
        NULLIF(TRIM(nutriments_estimated), '') AS nutriments_estimated,
        NULLIF(TRIM(nutrition_data), '') AS nutrition_data,
        NULLIF(TRIM(nutrition_data_per), '') AS nutrition_data_per,
        NULLIF(TRIM(nutrition_data_prepared), '') AS nutrition_data_prepared,
        NULLIF(TRIM(nutrition_data_prepared_per), '') AS nutrition_data_prepared_per

    FROM read_parquet('s3://bronze/openfoodfacts/raw/produits.parquet')
    WHERE
        NULLIF(TRIM(code), '') IS NOT NULL
        AND NULLIF(TRIM(product_name), '') IS NOT NULL
""")

print("Table produits_silver créée !")

Table produits_silver créée !


In [51]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT code) AS codes_distincts,
        COUNT(*) FILTER (
            WHERE nova_group NOT BETWEEN 1 AND 4
              AND nova_group IS NOT NULL
        ) AS nova_invalides,
        COUNT(*) FILTER (
            WHERE nutriscore_grade NOT IN ('a', 'b', 'c', 'd', 'e')
              AND nutriscore_grade IS NOT NULL
        ) AS nutriscores_invalides,
        COUNT(*) FILTER (
            WHERE serving_quantity < 0
        ) AS quantites_negatives,
        COUNT(*) FILTER (
            WHERE NOT json_valid(nutriments)
        ) AS json_invalides
    FROM produits_silver
""").df()

,total_lignes,codes_distincts,nova_invalides,nutriscores_invalides,quantites_negatives,json_invalides
0,100,100,0,1,0,0


In [52]:
con.execute("""
    COPY produits_silver
    TO 's3://silver/openfoodfacts/produits.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Fichier produits.parquet exporté dans Silver !")

Fichier produits.parquet exporté dans Silver !


In [53]:
con.execute("""
    SELECT *
    FROM read_parquet('s3://bronze/openprices/raw/page.parquet')
    LIMIT 1
""").df()

,textes,url
0,"["" Open Prices"", ""EXPLORE"", ""CONTRIBUTE"", ""SIG...",https://prices.openfoodfacts.org/


In [54]:
con.execute("""
    SELECT
        json_valid(textes) AS json_valide,
        LENGTH(textes) AS longueur_texte,
        LEFT(textes, 3000) AS apercu_textes
    FROM read_parquet('s3://bronze/openprices/raw/page.parquet')
""").df()

,json_valide,longueur_texte,apercu_textes
0,True,4121,"["" Open Prices"", ""EXPLORE"", ""CONTRIBUTE"", ""SIG..."


In [55]:
import requests

fichiers = ["prices", "proofs", "locations", "products"]

for nom in fichiers:
    url = f"https://prices.openfoodfacts.org/data/{nom}.jsonl.gz"
    response = requests.head(url, allow_redirects=True, timeout=30)

    taille = response.headers.get("Content-Length")

    if response.ok and taille:
        taille_mo = int(taille) / (1024 * 1024)
        print(f"{nom:12} | Disponible | {taille_mo:.2f} Mo")
    else:
        print(f"{nom:12} | Statut HTTP : {response.status_code}")

prices       | Disponible | 21.86 Mo
proofs       | Disponible | 9.35 Mo
locations    | Disponible | 1.20 Mo
products     | Disponible | 19.50 Mo


In [56]:
import requests
import boto3
import os

# Connexion à RustFS
s3 = boto3.client(
    "s3",
    endpoint_url="http://rustfs:9000",
    aws_access_key_id=os.getenv("S3_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("S3_SECRET_KEY"),
    region_name="us-east-1"
)

fichiers = ["prices", "proofs", "locations", "products"]

for nom in fichiers:
    url = f"https://prices.openfoodfacts.org/data/{nom}.jsonl.gz"
    chemin_local = f"/tmp/{nom}.jsonl.gz"
    chemin_bronze = f"openprices/raw/{nom}.jsonl.gz"

    print(f"Téléchargement de {nom}...")

    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()

    with open(chemin_local, "wb") as fichier:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                fichier.write(chunk)

    s3.upload_file(
        chemin_local,
        "bronze",
        chemin_bronze
    )

    taille = os.path.getsize(chemin_local) / (1024 * 1024)
    print(f"{nom} ajouté dans Bronze : {taille:.2f} Mo")

print("\\nTéléchargement des quatre fichiers terminé !")

Téléchargement de prices...
prices ajouté dans Bronze : 21.86 Mo
Téléchargement de proofs...
proofs ajouté dans Bronze : 9.35 Mo
Téléchargement de locations...
locations ajouté dans Bronze : 1.20 Mo
Téléchargement de products...
products ajouté dans Bronze : 19.50 Mo
\nTéléchargement des quatre fichiers terminé !


In [57]:
response = s3.list_objects_v2(
    Bucket="bronze",
    Prefix="openprices/raw/"
)

for obj in response.get("Contents", []):
    print(
        f"{obj['Key']} | "
        f"{obj['Size'] / (1024 * 1024):.2f} Mo"
    )

openprices/raw/locations.jsonl.gz | 1.20 Mo
openprices/raw/page.parquet | 0.00 Mo
openprices/raw/prices.jsonl.gz | 21.86 Mo
openprices/raw/products.jsonl.gz | 19.50 Mo
openprices/raw/proofs.jsonl.gz | 9.35 Mo


In [58]:
response = s3.list_objects_v2(
    Bucket="bronze",
    Prefix="openprices/raw/"
)

for obj in response.get("Contents", []):
    print(
        f"{obj['Key']} | "
        f"{obj['Size'] / (1024 * 1024):.2f} Mo"
    )

openprices/raw/locations.jsonl.gz | 1.20 Mo
openprices/raw/page.parquet | 0.00 Mo
openprices/raw/prices.jsonl.gz | 21.86 Mo
openprices/raw/products.jsonl.gz | 19.50 Mo
openprices/raw/proofs.jsonl.gz | 9.35 Mo


In [59]:
fichiers = ["prices", "proofs", "locations", "products"]

for nom in fichiers:
    print(f"\n{'='*50}")
    print(f"STRUCTURE : {nom.upper()}")
    print("="*50)

    schema = con.execute(f"""
        DESCRIBE
        SELECT *
        FROM read_json_auto(
            's3://bronze/openprices/raw/{nom}.jsonl.gz'
        )
    """).df()

    display(schema)


STRUCTURE : PRICES


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,product_id,BIGINT,YES,None,None,None
2,location_id,BIGINT,YES,None,None,None
3,proof_id,BIGINT,YES,None,None,None
4,type,VARCHAR,YES,None,None,None
5,product_code,VARCHAR,YES,None,None,None
6,product_name,VARCHAR,YES,None,None,None
7,category_tag,VARCHAR,YES,None,None,None
8,labels_tags,VARCHAR[],YES,None,None,None
9,origins_tags,VARCHAR[],YES,None,None,None



STRUCTURE : PROOFS


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,location_id,BIGINT,YES,None,None,None
2,file_path,VARCHAR,YES,None,None,None
3,mimetype,VARCHAR,YES,None,None,None
4,type,VARCHAR,YES,None,None,None
5,image_md5_hash,VARCHAR,YES,None,None,None
6,image_thumb_path,VARCHAR,YES,None,None,None
7,location_osm_id,BIGINT,YES,None,None,None
8,location_osm_type,VARCHAR,YES,None,None,None
9,date,DATE,YES,None,None,None



STRUCTURE : LOCATIONS


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,osm_brand_logo_url,VARCHAR,YES,None,None,None
2,type,VARCHAR,YES,None,None,None
3,osm_id,BIGINT,YES,None,None,None
4,osm_type,VARCHAR,YES,None,None,None
5,osm_name,VARCHAR,YES,None,None,None
6,osm_display_name,VARCHAR,YES,None,None,None
7,osm_tag_key,VARCHAR,YES,None,None,None
8,osm_tag_value,VARCHAR,YES,None,None,None
9,osm_tags,VARCHAR[],YES,None,None,None



STRUCTURE : PRODUCTS


,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,code,VARCHAR,YES,None,None,None
2,source,VARCHAR,YES,None,None,None
3,source_last_synced,TIMESTAMP WITH TIME ZONE,YES,None,None,None
4,product_name,VARCHAR,YES,None,None,None
5,image_url,VARCHAR,YES,None,None,None
6,product_quantity,BIGINT,YES,None,None,None
7,product_quantity_unit,VARCHAR,YES,None,None,None
8,quantity,VARCHAR,YES,None,None,None
9,categories_tags,VARCHAR[],YES,None,None,None


In [60]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (
            WHERE id IS NULL
        ) AS ids_manquants,
        COUNT(*) FILTER (
            WHERE code IS NULL OR TRIM(code) = ''
        ) AS codes_manquants,
        COUNT(*) FILTER (
            WHERE product_name IS NULL OR TRIM(product_name) = ''
        ) AS noms_manquants,
        COUNT(*) FILTER (
            WHERE nova_group IS NOT NULL
              AND nova_group NOT BETWEEN 1 AND 4
        ) AS nova_invalides,
        COUNT(*) FILTER (
            WHERE price_count < 0
        ) AS prix_negatifs
    FROM read_json_auto(
        's3://bronze/openprices/raw/products.jsonl.gz'
    )
""").df()

,total_lignes,ids_distincts,ids_manquants,codes_manquants,noms_manquants,nova_invalides,prix_negatifs
0,149951,149951,0,0,23382,0,0


In [61]:
con.execute("""
    SELECT
        id,
        code,
        source,
        brands,
        categories_tags,
        price_count
    FROM read_json_auto(
        's3://bronze/openprices/raw/products.jsonl.gz'
    )
    WHERE product_name IS NULL
       OR TRIM(product_name) = ''
    LIMIT 10
""").df()

,id,code,source,brands,categories_tags,price_count
0,3863436,01499315,None,None,[],1
1,3763578,7090018978210,None,None,[],5
2,3838989,5054563206978,obf,None,[],3
3,4633140,5000394145573,None,None,[],1
4,4322914,3254567945886,None,None,[],1
5,3846384,7391835917896,None,None,[],1
6,4261107,3760373420412,None,None,[],1
7,4714492,8006530191975,None,None,[],1
8,3740261,5900471013279,None,None,[],1
9,3851288,8003340805832,off,Lindt,"[en:snacks, en:sweet-snacks, en:cocoa-and-its-...",2


In [62]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    )
""").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,product_id,BIGINT,YES,None,None,None
2,location_id,BIGINT,YES,None,None,None
3,proof_id,BIGINT,YES,None,None,None
4,type,VARCHAR,YES,None,None,None
5,product_code,VARCHAR,YES,None,None,None
6,product_name,VARCHAR,YES,None,None,None
7,category_tag,VARCHAR,YES,None,None,None
8,labels_tags,VARCHAR[],YES,None,None,None
9,origins_tags,VARCHAR[],YES,None,None,None


In [63]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE id IS NULL) AS ids_manquants,
        COUNT(*) FILTER (
            WHERE TRY_CAST(price AS DOUBLE) IS NULL
        ) AS prix_non_numeriques,
        COUNT(*) FILTER (
            WHERE TRY_CAST(price AS DOUBLE) <= 0
        ) AS prix_nuls_ou_negatifs,
        COUNT(*) FILTER (
            WHERE product_id IS NULL
        ) AS produits_manquants,
        COUNT(*) FILTER (
            WHERE location_id IS NULL
        ) AS lieux_manquants,
        COUNT(*) FILTER (
            WHERE date IS NULL
        ) AS dates_manquantes,
        COUNT(*) FILTER (
            WHERE currency IS NULL OR TRIM(currency) = ''
        ) AS devises_manquantes
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    )
""").df()

,total_lignes,ids_distincts,ids_manquants,prix_non_numeriques,prix_nuls_ou_negatifs,produits_manquants,lieux_manquants,dates_manquantes,devises_manquantes
0,344523,344523,0,0,13,11060,2,185,170


In [64]:
con.execute("""
    SELECT
        id,
        product_id,
        location_id,
        price,
        currency,
        date,
        type
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    )
    WHERE
        TRY_CAST(price AS DOUBLE) <= 0
        OR product_id IS NULL
        OR location_id IS NULL
        OR date IS NULL
        OR currency IS NULL
        OR TRIM(currency) = ''
    LIMIT 20
""").df()

,id,product_id,location_id,price,currency,date,type
0,43283,<NA>,127,3.900,EUR,2024-10-28,CATEGORY
1,29746,<NA>,469,14.950,EUR,2024-07-05,CATEGORY
2,43286,<NA>,127,1.990,EUR,2024-10-28,CATEGORY
3,43292,<NA>,127,2.500,EUR,2024-10-28,CATEGORY
4,53737,<NA>,100,29.900,PLN,2024-11-28,CATEGORY
5,37919,<NA>,245,4.350,PLN,2024-10-05,CATEGORY
6,34084,1826557,1027,9.950,None,NaT,PRODUCT
7,53738,<NA>,100,5.980,PLN,2024-11-28,CATEGORY
8,43475,<NA>,1377,5.200,EUR,2024-11-01,CATEGORY
9,42339,<NA>,34,1.990,EUR,2024-10-26,CATEGORY


In [65]:
con.execute("""
    SELECT
        type,
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE product_id IS NULL) AS produits_manquants,
        COUNT(*) FILTER (WHERE location_id IS NULL) AS lieux_manquants,
        COUNT(*) FILTER (WHERE date IS NULL) AS dates_manquantes,
        COUNT(*) FILTER (
            WHERE currency IS NULL OR TRIM(currency) = ''
        ) AS devises_manquantes,
        COUNT(*) FILTER (
            WHERE TRY_CAST(price AS DOUBLE) <= 0
        ) AS prix_nuls_ou_negatifs
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    )
    GROUP BY type
    ORDER BY total DESC
""").df()

,type,total,produits_manquants,lieux_manquants,dates_manquantes,devises_manquantes,prix_nuls_ou_negatifs
0,PRODUCT,333823,360,2,185,170,12
1,CATEGORY,10700,10700,0,0,0,1


In [66]:
con.execute("""
    SELECT
        id,
        type,
        product_id,
        price,
        price_without_discount,
        currency,
        date
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    )
    WHERE TRY_CAST(price AS DOUBLE) <= 0
""").df()

,id,type,product_id,price,price_without_discount,currency,date
0,88331,PRODUCT,3782594,0.000,None,EUR,2025-03-09
1,88332,PRODUCT,3862750,0.000,None,EUR,2025-03-09
2,33580,PRODUCT,308570,0.000,None,EUR,2023-02-11
3,84026,PRODUCT,3843182,0.000,None,EUR,2021-10-03
4,84004,PRODUCT,3843182,0.000,None,EUR,2021-10-08
5,100425,PRODUCT,210871,0.000,1.190,EUR,2025-04-22
6,100797,PRODUCT,2628105,0.000,2.990,EUR,2025-04-22
7,100798,PRODUCT,1754440,0.000,7.990,EUR,2025-04-22
8,122661,PRODUCT,3123157,0.000,3.150,EUR,2025-06-28
9,43141,CATEGORY,<NA>,0.000,3.900,SEK,2024-09-17


In [67]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_json_auto(
        's3://bronze/openprices/raw/locations.jsonl.gz'
    )
""").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,osm_brand_logo_url,VARCHAR,YES,None,None,None
2,type,VARCHAR,YES,None,None,None
3,osm_id,BIGINT,YES,None,None,None
4,osm_type,VARCHAR,YES,None,None,None
5,osm_name,VARCHAR,YES,None,None,None
6,osm_display_name,VARCHAR,YES,None,None,None
7,osm_tag_key,VARCHAR,YES,None,None,None
8,osm_tag_value,VARCHAR,YES,None,None,None
9,osm_tags,VARCHAR[],YES,None,None,None


In [68]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE id IS NULL) AS ids_manquants,
        COUNT(*) FILTER (
            WHERE osm_lat IS NULL OR TRY_CAST(osm_lat AS DOUBLE) IS NULL
        ) AS latitudes_invalides,
        COUNT(*) FILTER (
            WHERE osm_lon IS NULL OR TRY_CAST(osm_lon AS DOUBLE) IS NULL
        ) AS longitudes_invalides,
        COUNT(*) FILTER (
            WHERE osm_address_country IS NULL
        ) AS pays_manquants,
        COUNT(*) FILTER (
            WHERE osm_address_city IS NULL
        ) AS villes_manquantes
    FROM read_json_auto(
        's3://bronze/openprices/raw/locations.jsonl.gz'
    )
""").df()

,total_lignes,ids_distincts,ids_manquants,latitudes_invalides,longitudes_invalides,pays_manquants,villes_manquantes
0,7926,7926,0,191,191,191,254


In [69]:
con.execute("""
    SELECT
        id,
        osm_name,
        osm_address_city,
        osm_address_country,
        osm_lat,
        osm_lon
    FROM read_json_auto(
        's3://bronze/openprices/raw/locations.jsonl.gz'
    )
    WHERE
        TRY_CAST(osm_lat AS DOUBLE) IS NULL
        OR TRY_CAST(osm_lon AS DOUBLE) IS NULL
    LIMIT 15
""").df()

,id,osm_name,osm_address_city,osm_address_country,osm_lat,osm_lon
0,2492,None,None,None,None,None
1,4839,None,None,None,None,None
2,4803,None,None,None,None,None
3,1514,None,None,None,None,None
4,1775,None,None,None,None,None
5,7818,None,None,None,None,None
6,1513,None,None,None,None,None
7,7380,None,None,None,None,None
8,5385,None,None,None,None,None
9,2863,None,None,None,None,None


In [70]:
con.execute("""
    SELECT
        COUNT(*) AS prix_avec_lieu,
        COUNT(*) FILTER (
            WHERE l.id IS NOT NULL
        ) AS lieux_correspondants,
        COUNT(*) FILTER (
            WHERE l.id IS NULL
        ) AS lieux_introuvables
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    ) p
    LEFT JOIN read_json_auto(
        's3://bronze/openprices/raw/locations.jsonl.gz'
    ) l
        ON p.location_id = l.id
    WHERE p.location_id IS NOT NULL
""").df()

,prix_avec_lieu,lieux_correspondants,lieux_introuvables
0,344521,344521,0


In [71]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_json_auto(
        's3://bronze/openprices/raw/proofs.jsonl.gz'
    )
""").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,location_id,BIGINT,YES,None,None,None
2,file_path,VARCHAR,YES,None,None,None
3,mimetype,VARCHAR,YES,None,None,None
4,type,VARCHAR,YES,None,None,None
5,image_md5_hash,VARCHAR,YES,None,None,None
6,image_thumb_path,VARCHAR,YES,None,None,None
7,location_osm_id,BIGINT,YES,None,None,None
8,location_osm_type,VARCHAR,YES,None,None,None
9,date,DATE,YES,None,None,None


In [72]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE id IS NULL) AS ids_manquants,
        COUNT(*) FILTER (WHERE location_id IS NULL) AS lieux_manquants,
        COUNT(*) FILTER (WHERE file_path IS NULL) AS fichiers_manquants,
        COUNT(*) FILTER (WHERE date IS NULL) AS dates_manquantes,
        COUNT(*) FILTER (WHERE mimetype IS NULL) AS mimetypes_manquants
    FROM read_json_auto(
        's3://bronze/openprices/raw/proofs.jsonl.gz'
    )
""").df()

,total_lignes,ids_distincts,ids_manquants,lieux_manquants,fichiers_manquants,dates_manquantes,mimetypes_manquants
0,134132,134132,0,1384,0,1527,0


In [73]:
con.execute("""
    SELECT
        COUNT(*) AS prix_avec_preuve,
        COUNT(*) FILTER (
            WHERE pr.id IS NOT NULL
        ) AS preuves_correspondantes,
        COUNT(*) FILTER (
            WHERE pr.id IS NULL
        ) AS preuves_introuvables
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    ) p
    LEFT JOIN read_json_auto(
        's3://bronze/openprices/raw/proofs.jsonl.gz'
    ) pr
        ON p.proof_id = pr.id
    WHERE p.proof_id IS NOT NULL
""").df()

,prix_avec_preuve,preuves_correspondantes,preuves_introuvables
0,344394,344394,0


In [74]:
con.execute("""
    SELECT
        COUNT(*) AS prix_avec_produit,
        COUNT(*) FILTER (
            WHERE pr.id IS NOT NULL
        ) AS produits_correspondants,
        COUNT(*) FILTER (
            WHERE pr.id IS NULL
        ) AS produits_introuvables
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    ) p
    LEFT JOIN read_json_auto(
        's3://bronze/openprices/raw/products.jsonl.gz'
    ) pr
        ON p.product_id = pr.id
    WHERE p.product_id IS NOT NULL
""").df()

,prix_avec_produit,produits_correspondants,produits_introuvables
0,333463,333463,0


In [75]:
con.execute("""
    SELECT
        currency,
        COUNT(*) AS nombre_prix,
        MIN(TRY_CAST(price AS DOUBLE)) AS prix_min,
        MAX(TRY_CAST(price AS DOUBLE)) AS prix_max,
        AVG(TRY_CAST(price AS DOUBLE)) AS prix_moyen
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    )
    WHERE TRY_CAST(price AS DOUBLE) > 0
    GROUP BY currency
    ORDER BY nombre_prix DESC
""").df()

,currency,nombre_prix,prix_min,prix_max,prix_moyen
0,EUR,251100,0.01,2102.00,4.056039
1,USD,47954,0.07,3199.99,7.947313
2,NOK,25135,1.00,899.00,44.283254
3,SEK,5879,2.00,1999.00,34.136965
4,PLN,2343,0.25,124.99,9.872390
...,...,...,...,...,...
90,OMR,1,0.46,0.46,0.460000
91,MKD,1,57.00,57.00,57.000000
92,BOB,1,10.50,10.50,10.500000
93,RWF,1,1100.00,1100.00,1100.000000


In [76]:
con.execute("""
    CREATE OR REPLACE TABLE prices_silver AS
    SELECT
        id,
        product_id,
        location_id,
        proof_id,
        type,
        NULLIF(TRIM(product_code), '') AS product_code,
        NULLIF(TRIM(product_name), '') AS product_name,
        NULLIF(TRIM(category_tag), '') AS category_tag,
        labels_tags,
        origins_tags,
        TRY_CAST(price AS DOUBLE) AS price,
        price_is_discounted,
        TRY_CAST(price_without_discount AS DOUBLE)
            AS price_without_discount,
        NULLIF(TRIM(discount_type), '') AS discount_type,
        NULLIF(TRIM(price_per), '') AS price_per,
        NULLIF(TRIM(currency), '') AS currency,
        location_osm_id,
        location_osm_type,
        date,
        NULLIF(TRIM(receipt_quantity), '') AS receipt_quantity,
        NULLIF(TRIM(owner_comment), '') AS owner_comment,
        NULLIF(TRIM(owner), '') AS owner,
        NULLIF(TRIM(source), '') AS source,
        tags,
        created,
        updated,
        duplicate_of
    FROM read_json_auto(
        's3://bronze/openprices/raw/prices.jsonl.gz'
    )
    WHERE
        id IS NOT NULL
        AND TRY_CAST(price AS DOUBLE) > 0
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [77]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE price <= 0) AS prix_invalides,
        COUNT(*) FILTER (WHERE currency IS NULL) AS devises_manquantes,
        COUNT(*) FILTER (WHERE product_id IS NULL) AS produits_manquants,
        COUNT(*) FILTER (WHERE location_id IS NULL) AS lieux_manquants
    FROM prices_silver
""").df()

,total_lignes,ids_distincts,prix_invalides,devises_manquantes,produits_manquants,lieux_manquants
0,344510,344510,0,170,11059,2


In [78]:
con.execute("""
    COPY prices_silver
    TO 's3://silver/openprices/prices.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [79]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE id IS NULL) AS ids_manquants,
        COUNT(*) FILTER (
            WHERE code IS NULL OR TRIM(code) = ''
        ) AS codes_manquants,
        COUNT(*) FILTER (
            WHERE product_name IS NULL OR TRIM(product_name) = ''
        ) AS noms_manquants,
        COUNT(*) FILTER (
            WHERE nova_group IS NOT NULL
              AND nova_group NOT BETWEEN 1 AND 4
        ) AS nova_invalides,
        COUNT(*) FILTER (WHERE price_count < 0) AS prix_negatifs
    FROM read_json_auto(
        's3://bronze/openprices/raw/products.jsonl.gz'
    )
""").df()

,total_lignes,ids_distincts,ids_manquants,codes_manquants,noms_manquants,nova_invalides,prix_negatifs
0,149951,149951,0,0,23382,0,0


In [80]:
con.execute("""
    CREATE OR REPLACE TABLE products_silver AS
    SELECT
        id,
        NULLIF(TRIM(code), '') AS code,
        NULLIF(TRIM(source), '') AS source,
        source_last_synced,
        NULLIF(TRIM(product_name), '') AS product_name,
        NULLIF(TRIM(image_url), '') AS image_url,
        product_quantity,
        NULLIF(TRIM(product_quantity_unit), '') AS product_quantity_unit,
        NULLIF(TRIM(quantity), '') AS quantity,
        categories_tags,
        NULLIF(TRIM(brands), '') AS brands,
        brands_tags,
        labels_tags,
        LOWER(NULLIF(TRIM(nutriscore_grade), '')) AS nutriscore_grade,
        LOWER(NULLIF(TRIM(ecoscore_grade), '')) AS ecoscore_grade,
        CASE
            WHEN nova_group BETWEEN 1 AND 4
            THEN nova_group
            ELSE NULL
        END AS nova_group,
        NULLIF(TRIM(creator), '') AS creator,
        unique_scans_n,
        price_count,
        price_currency_count,
        location_count,
        location_type_osm_country_count,
        user_count,
        proof_count,
        created,
        updated
    FROM read_json_auto(
        's3://bronze/openprices/raw/products.jsonl.gz'
    )
    WHERE id IS NOT NULL
""")

In [81]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE id IS NULL) AS ids_manquants,
        COUNT(*) FILTER (WHERE product_name IS NULL) AS noms_manquants,
        COUNT(*) FILTER (
            WHERE nova_group IS NOT NULL
              AND nova_group NOT BETWEEN 1 AND 4
        ) AS nova_invalides
    FROM products_silver
""").df()

,total_lignes,ids_distincts,ids_manquants,noms_manquants,nova_invalides
0,149951,149951,0,23382,0


In [82]:
con.execute("""
    COPY products_silver
    TO 's3://silver/openprices/products.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [83]:
con.execute("""
    CREATE OR REPLACE TABLE locations_silver AS
    SELECT
        id,
        NULLIF(TRIM(osm_brand_logo_url), '') AS osm_brand_logo_url,
        NULLIF(TRIM(type), '') AS type,
        osm_id,
        NULLIF(TRIM(osm_type), '') AS osm_type,
        NULLIF(TRIM(osm_name), '') AS osm_name,
        NULLIF(TRIM(osm_display_name), '') AS osm_display_name,
        NULLIF(TRIM(osm_tag_key), '') AS osm_tag_key,
        NULLIF(TRIM(osm_tag_value), '') AS osm_tag_value,
        osm_tags,
        NULLIF(TRIM(osm_brand), '') AS osm_brand,
        NULLIF(TRIM(osm_address_postcode), '') AS osm_address_postcode,
        NULLIF(TRIM(osm_address_city), '') AS osm_address_city,
        NULLIF(TRIM(osm_address_country), '') AS osm_address_country,
        NULLIF(TRIM(osm_address_country_code), '') AS osm_address_country_code,
        TRY_CAST(osm_lat AS DOUBLE) AS osm_lat,
        TRY_CAST(osm_lon AS DOUBLE) AS osm_lon,
        osm_version,
        osm_version_date,
        NULLIF(TRIM(website_url), '') AS website_url,
        tags,
        price_count,
        user_count,
        product_count,
        proof_count,
        NULLIF(TRIM(source), '') AS source,
        created,
        updated
    FROM read_json_auto(
        's3://bronze/openprices/raw/locations.jsonl.gz'
    )
    WHERE id IS NOT NULL
""")

In [84]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE id IS NULL) AS ids_manquants,
        COUNT(*) FILTER (
            WHERE osm_lat IS NOT NULL
              AND (osm_lat < -90 OR osm_lat > 90)
        ) AS latitudes_hors_limites,
        COUNT(*) FILTER (
            WHERE osm_lon IS NOT NULL
              AND (osm_lon < -180 OR osm_lon > 180)
        ) AS longitudes_hors_limites,
        COUNT(*) FILTER (WHERE osm_lat IS NULL) AS latitudes_manquantes,
        COUNT(*) FILTER (WHERE osm_lon IS NULL) AS longitudes_manquantes
    FROM locations_silver
""").df()

,total_lignes,ids_distincts,ids_manquants,latitudes_hors_limites,longitudes_hors_limites,latitudes_manquantes,longitudes_manquantes
0,7926,7926,0,0,0,191,191


In [85]:
con.execute("""
    COPY locations_silver
    TO 's3://silver/openprices/locations.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [86]:
con.execute("""
    SELECT
        receipt_price_total,
        receipt_online_delivery_costs
    FROM read_json_auto(
        's3://bronze/openprices/raw/proofs.jsonl.gz'
    )
    WHERE receipt_price_total IS NOT NULL
       OR receipt_online_delivery_costs IS NOT NULL
    LIMIT 10
""").df()

,receipt_price_total,receipt_online_delivery_costs
0,3.880,None
1,42.480,None
2,13.440,None
3,10.950,None
4,22.600,None
5,11.680,None
6,9.300,None
7,26.110,None
8,8.340,None
9,6.070,None


In [87]:
con.execute("""
    CREATE OR REPLACE TABLE proofs_silver AS
    SELECT
        id,
        location_id,
        NULLIF(TRIM(file_path), '') AS file_path,
        NULLIF(TRIM(mimetype), '') AS mimetype,
        NULLIF(TRIM(type), '') AS type,
        NULLIF(TRIM(image_md5_hash), '') AS image_md5_hash,
        NULLIF(TRIM(image_thumb_path), '') AS image_thumb_path,
        location_osm_id,
        NULLIF(TRIM(location_osm_type), '') AS location_osm_type,
        date,
        NULLIF(TRIM(currency), '') AS currency,
        receipt_price_count,
        TRY_CAST(receipt_price_total AS DOUBLE) AS receipt_price_total,
        TRY_CAST(receipt_online_delivery_costs AS DOUBLE)
            AS receipt_online_delivery_costs,
        ready_for_price_tag_validation,
        owner_consumption,
        NULLIF(TRIM(owner_comment), '') AS owner_comment,
        draft,
        price_count,
        prediction_count,
        NULLIF(TRIM(owner), '') AS owner,
        NULLIF(TRIM(source), '') AS source,
        tags,
        created,
        updated
    FROM read_json_auto(
        's3://bronze/openprices/raw/proofs.jsonl.gz'
    )
    WHERE id IS NOT NULL
""")

In [88]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE id IS NULL) AS ids_manquants,
        COUNT(*) FILTER (WHERE location_id IS NULL) AS lieux_manquants,
        COUNT(*) FILTER (WHERE file_path IS NULL) AS fichiers_manquants,
        COUNT(*) FILTER (
            WHERE receipt_price_total IS NULL
        ) AS montants_totaux_manquants,
        COUNT(*) FILTER (
            WHERE receipt_online_delivery_costs IS NULL
        ) AS frais_livraison_manquants
    FROM proofs_silver
""").df()

,total_lignes,ids_distincts,ids_manquants,lieux_manquants,fichiers_manquants,montants_totaux_manquants,frais_livraison_manquants
0,134132,134132,0,1384,0,131127,134047


In [89]:
con.execute("""
    COPY proofs_silver
    TO 's3://silver/openprices/proofs.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [90]:
con.execute("""
    SELECT 'products' AS table_name, COUNT(*) AS lignes
    FROM read_parquet('s3://silver/openprices/products.parquet')

    UNION ALL

    SELECT 'prices', COUNT(*)
    FROM read_parquet('s3://silver/openprices/prices.parquet')

    UNION ALL

    SELECT 'locations', COUNT(*)
    FROM read_parquet('s3://silver/openprices/locations.parquet')

    UNION ALL

    SELECT 'proofs', COUNT(*)
    FROM read_parquet('s3://silver/openprices/proofs.parquet')
""").df()

,table_name,lignes
0,products,149951
1,prices,344510
2,locations,7926
3,proofs,134132


In [91]:
from pathlib import Path

dossier = Path("/home/jovyan/work")

fichiers = [
    f for f in dossier.rglob("*")
    if f.is_file()
    and ("rappelconso" in str(f).lower()
         or "rappel_conso" in str(f).lower())
]

for f in fichiers:
    print(f)

/home/jovyan/work/importation/rappelconso.py
/home/jovyan/work/script/rappelconso_quotidien.sh
/home/jovyan/work/importation/__pycache__/rappelconso.cpython-313.pyc
/home/jovyan/work/data/rappelconso/collecte.sqlite
/home/jovyan/work/data/rappelconso/rappels.json
/home/jovyan/work/data/rappelconso/rappels.parquet


In [92]:
for obj in s3.list_objects_v2(
    Bucket="bronze",
    Prefix="rappelconso/"
).get("Contents", []):
    print(obj["Key"], round(obj["Size"] / 1024, 2), "Ko")

rappelconso/raw/rappels.parquet 6378.26 Ko


In [93]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        's3://bronze/rappelconso/raw/rappels.parquet'
    )
""").df()

,column_name,column_type,null,key,default,extra
0,categorie_produit,VARCHAR,YES,None,None,None
1,conditionnements,VARCHAR,YES,None,None,None
2,conduites_a_tenir_par_le_consommateur,VARCHAR,YES,None,None,None
3,date_date_fin_commercialisation,VARCHAR,YES,None,None,None
4,date_de_fin_de_la_procedure_de_rappel,VARCHAR,YES,None,None,None
5,date_debut_commercialisation,VARCHAR,YES,None,None,None
6,date_publication,VARCHAR,YES,None,None,None
7,description_complementaire_risque,VARCHAR,YES,None,None,None
8,distributeurs,VARCHAR,YES,None,None,None
9,id,VARCHAR,YES,None,None,None


In [94]:
con.execute("""
    SELECT *
    FROM read_parquet(
        's3://bronze/rappelconso/raw/rappels.parquet'
    )
    LIMIT 5
""").df()

,categorie_produit,conditionnements,conduites_a_tenir_par_le_consommateur,date_date_fin_commercialisation,date_de_fin_de_la_procedure_de_rappel,date_debut_commercialisation,date_publication,description_complementaire_risque,distributeurs,id,...,nature_juridique_rappel,numero_contact,numero_fiche,numero_version,preconisations_sanitaires,rappel_guid,risques_encourus,sous_categorie_produit,temperature_conservation,zone_geographique_de_vente
0,automobiles et moyens de déplacement,None,None,None,None,None,2018-01-19T00:00:00+00:00,None,None,49194,...,None,None,a12/0043/18,0,None,ecf3678e21e68f9bb76038e161a8de2d4d6d0ad2,blessures,"automobiles, motos, scooters",None,None
1,automobiles et moyens de déplacement,None,None,None,None,None,2018-01-19T00:00:00+00:00,None,None,49195,...,None,None,a12/0042/18,0,None,f6d48cd5356a6e554f2eacfa4db244dd08f5dfb0,blessures,"automobiles, motos, scooters",None,None
2,automobiles et moyens de déplacement,None,None,None,None,None,2018-01-26T00:00:00+00:00,None,None,49192,...,None,None,a12/0093/18,0,None,4a6b3ffe96259d97af1662344ffc535721b4c919,blessures,"automobiles, motos, scooters",None,None
3,automobiles et moyens de déplacement,None,None,None,None,None,2018-01-26T00:00:00+00:00,None,None,49193,...,None,None,a12/0092/18,0,None,e90a4b1399829aaaba53ed167ff39df1882611e2,blessures,"automobiles, motos, scooters",None,None
4,automobiles et moyens de déplacement,None,None,None,None,None,2018-02-16T00:00:00+00:00,None,None,49190,...,None,None,a12/0221/18,0,None,7318ae2c743121d699fdca4369d62c210c210865,blessures,"automobiles, motos, scooters",None,None


In [95]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,

        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (
            WHERE id IS NULL OR TRIM(id) = ''
        ) AS ids_manquants,

        COUNT(DISTINCT rappel_guid) AS guids_distincts,
        COUNT(*) FILTER (
            WHERE rappel_guid IS NULL OR TRIM(rappel_guid) = ''
        ) AS guids_manquants,

        COUNT(DISTINCT numero_fiche) AS numeros_fiche_distincts,

        COUNT(*) FILTER (
            WHERE date_publication IS NULL
               OR TRY_CAST(date_publication AS TIMESTAMPTZ) IS NULL
        ) AS dates_publication_invalides

    FROM read_parquet(
        's3://bronze/rappelconso/raw/rappels.parquet'
    )
""").df()

,total_lignes,ids_distincts,ids_manquants,guids_distincts,guids_manquants,numeros_fiche_distincts,dates_publication_invalides
0,19260,19260,0,19260,0,19260,0


In [96]:
con.execute("""
    SELECT
        MIN(TRY_CAST(date_publication AS TIMESTAMPTZ))
            AS premiere_publication,
        MAX(TRY_CAST(date_publication AS TIMESTAMPTZ))
            AS derniere_publication,
        COUNT(*) FILTER (
            WHERE numero_fiche IS NULL OR TRIM(numero_fiche) = ''
        ) AS numeros_fiche_manquants,
        COUNT(DISTINCT numero_fiche) AS numeros_fiche_distincts
    FROM read_parquet(
        's3://bronze/rappelconso/raw/rappels.parquet'
    )
""").df()

,premiere_publication,derniere_publication,numeros_fiche_manquants,numeros_fiche_distincts
0,2018-01-19 00:00:00+00:00,2026-11-17 09:50:10+00:00,0,19260


In [97]:
con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE date_debut_commercialisation IS NOT NULL
              AND TRY_CAST(date_debut_commercialisation AS TIMESTAMPTZ) IS NULL
        ) AS debut_invalides,

        COUNT(*) FILTER (
            WHERE date_date_fin_commercialisation IS NOT NULL
              AND TRY_CAST(date_date_fin_commercialisation AS TIMESTAMPTZ) IS NULL
        ) AS fin_commercialisation_invalides,

        COUNT(*) FILTER (
            WHERE date_de_fin_de_la_procedure_de_rappel IS NOT NULL
              AND TRY_CAST(date_de_fin_de_la_procedure_de_rappel AS TIMESTAMPTZ) IS NULL
        ) AS fin_procedure_invalides,

        COUNT(*) FILTER (
            WHERE date_debut_commercialisation IS NOT NULL
        ) AS debuts_renseignes,

        COUNT(*) FILTER (
            WHERE date_date_fin_commercialisation IS NOT NULL
        ) AS fins_commercialisation_renseignees,

        COUNT(*) FILTER (
            WHERE date_de_fin_de_la_procedure_de_rappel IS NOT NULL
        ) AS fins_procedure_renseignees

    FROM read_parquet(
        's3://bronze/rappelconso/raw/rappels.parquet'
    )
""").df()

,debut_invalides,fin_commercialisation_invalides,fin_procedure_invalides,debuts_renseignes,fins_commercialisation_renseignees,fins_procedure_renseignees
0,0,0,0,15076,14806,14090


In [98]:
con.execute("""
    SELECT
        categorie_produit,
        COUNT(*) AS nombre_rappels,
        COUNT(DISTINCT sous_categorie_produit) AS sous_categories
    FROM read_parquet(
        's3://bronze/rappelconso/raw/rappels.parquet'
    )
    GROUP BY categorie_produit
    ORDER BY nombre_rappels DESC
    LIMIT 15
""").df()

,categorie_produit,nombre_rappels,sous_categories
0,alimentation,14085,24
1,automobiles et moyens de déplacement,1925,6
2,bébés-enfants (hors alimentaire),1205,3
3,maison-habitat,518,8
4,"appareils électriques, outils",384,3
5,hygiène-beauté,359,4
6,"vêtements, mode, epi",328,3
7,autres,303,3
8,sports-loisirs,146,2
9,equipements de communication,7,1


In [99]:
con.execute("""
    SELECT
        conditionnements,
        distributeurs,
        identification_produits,
        liens_vers_les_images,
        zone_geographique_de_vente,
        modeles_ou_references
    FROM read_parquet(
        's3://bronze/rappelconso/raw/rappels.parquet'
    )
    WHERE
        conditionnements IS NOT NULL
        OR distributeurs IS NOT NULL
        OR identification_produits IS NOT NULL
        OR liens_vers_les_images IS NOT NULL
    LIMIT 5
""").df()

,conditionnements,distributeurs,identification_produits,liens_vers_les_images,zone_geographique_de_vente,modeles_ou_references
0,None,None,"[""vehicles manufactured between 5/4/2016 and 2...",None,None,3008 ii voiture de tourisme
1,None,None,"[""2008: manufactured between 2/10/2017 and 20/...",None,None,various models voiture de tourisme
2,None,None,"[""vehicles manufactured between 2/10/2017 and ...",None,None,"spacetourer, jumpy voiture de tourisme"
3,None,None,"[""vehicles manufactured between 2 october 2017...",None,None,"traveller, expert voiture de tourisme"
4,None,None,"[""vehicles manufactured between 5/12/2016 and ...",None,None,4008 voiture de tourisme


In [100]:
con.execute("""
    SELECT
        COUNT(*) FILTER (WHERE conditionnements IS NOT NULL AND TRIM(conditionnements) <> '') AS conditionnements_renseignes,
        COUNT(*) FILTER (WHERE distributeurs IS NOT NULL AND TRIM(distributeurs) <> '') AS distributeurs_renseignes,
        COUNT(*) FILTER (WHERE identification_produits IS NOT NULL AND TRIM(identification_produits) <> '') AS identification_renseignes,
        COUNT(*) FILTER (WHERE identification_produits IS NOT NULL AND json_valid(identification_produits)) AS identification_json_valides,
        COUNT(*) FILTER (WHERE liens_vers_les_images IS NOT NULL AND TRIM(liens_vers_les_images) <> '') AS images_renseignees,
        COUNT(*) FILTER (WHERE zone_geographique_de_vente IS NOT NULL AND TRIM(zone_geographique_de_vente) <> '') AS zones_renseignees,
        COUNT(*) FILTER (WHERE modeles_ou_references IS NOT NULL AND TRIM(modeles_ou_references) <> '') AS modeles_renseignes
    FROM read_parquet('s3://bronze/rappelconso/raw/rappels.parquet')
""").df()

,conditionnements_renseignes,distributeurs_renseignes,identification_renseignes,identification_json_valides,images_renseignees,zones_renseignees,modeles_renseignes
0,12606,17068,19260,19260,17317,17068,19260


In [101]:
con.execute("""
    SELECT
        json_type(identification_produits) AS type_identification,
        COUNT(*) AS nombre,
        MIN(json_array_length(identification_produits)) AS min_elements,
        MAX(json_array_length(identification_produits)) AS max_elements
    FROM read_parquet('s3://bronze/rappelconso/raw/rappels.parquet')
    GROUP BY type_identification
    ORDER BY nombre DESC
""").df()

,type_identification,nombre,min_elements,max_elements
0,ARRAY,19260,0,264


In [102]:
con.execute("""
    SELECT
        json_type(liens_vers_les_images) AS type_images,
        COUNT(*) AS nombre,
        MIN(json_array_length(liens_vers_les_images)) AS min_elements,
        MAX(json_array_length(liens_vers_les_images)) AS max_elements
    FROM read_parquet('s3://bronze/rappelconso/raw/rappels.parquet')
    WHERE liens_vers_les_images IS NOT NULL
      AND TRIM(liens_vers_les_images) <> ''
    GROUP BY type_images
    ORDER BY nombre DESC
""").df()

InvalidInputException: Invalid Input Error: Malformed JSON at byte 0 of input: unexpected character.  Input: "https://rappel.conso.gouv.fr/image/66a21860-c35..."

In [103]:
con.execute("""
    SELECT
        CASE
            WHEN json_valid(liens_vers_les_images) THEN json_type(liens_vers_les_images)
            WHEN liens_vers_les_images LIKE '[%](http%' THEN 'LIEN_MARKDOWN'
            WHEN liens_vers_les_images LIKE 'http%' THEN 'URL'
            ELSE 'AUTRE_TEXTE'
        END AS type_images,
        COUNT(*) AS nombre
    FROM read_parquet('s3://bronze/rappelconso/raw/rappels.parquet')
    WHERE liens_vers_les_images IS NOT NULL
      AND TRIM(liens_vers_les_images) <> ''
    GROUP BY type_images
    ORDER BY nombre DESC
""").df()

,type_images,nombre
0,URL,17317


In [104]:
con.execute("""
    SELECT
        COUNT(DISTINCT numero_version) AS versions_distinctes,
        MIN(TRY_CAST(numero_version AS INTEGER)) AS version_min,
        MAX(TRY_CAST(numero_version AS INTEGER)) AS version_max,
        COUNT(*) FILTER (
            WHERE numero_version IS NOT NULL
              AND TRY_CAST(numero_version AS INTEGER) IS NULL
        ) AS versions_non_numeriques,
        COUNT(*) FILTER (
            WHERE numero_version IS NULL OR TRIM(numero_version) = ''
        ) AS versions_manquantes
    FROM read_parquet('s3://bronze/rappelconso/raw/rappels.parquet')
""").df()

,versions_distinctes,version_min,version_max,versions_non_numeriques,versions_manquantes
0,8,0,7,0,0


In [105]:
con.execute("""
    COPY (
        SELECT * REPLACE (
            TRY_CAST(date_publication AS TIMESTAMPTZ) AS date_publication,
            TRY_CAST(date_debut_commercialisation AS TIMESTAMPTZ) AS date_debut_commercialisation,
            TRY_CAST(date_date_fin_commercialisation AS TIMESTAMPTZ) AS date_date_fin_commercialisation,
            TRY_CAST(date_de_fin_de_la_procedure_de_rappel AS TIMESTAMPTZ) AS date_de_fin_de_la_procedure_de_rappel,
            TRY_CAST(numero_version AS INTEGER) AS numero_version
        )
        FROM read_parquet('s3://bronze/rappelconso/raw/rappels.parquet')
    )
    TO 's3://silver/rappelconso/rappels.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [106]:
con.execute("""
    SELECT
        COUNT(*) AS total_lignes,
        COUNT(DISTINCT id) AS ids_distincts,
        COUNT(*) FILTER (WHERE date_publication IS NULL) AS dates_publication_nulles,
        COUNT(*) FILTER (WHERE numero_version IS NULL) AS versions_nulles,
        COUNT(*) FILTER (WHERE identification_produits IS NULL) AS identifications_nulles
    FROM read_parquet('s3://silver/rappelconso/rappels.parquet')
""").df()

,total_lignes,ids_distincts,dates_publication_nulles,versions_nulles,identifications_nulles
0,19260,19260,0,0,0


In [107]:
con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet('s3://silver/rappelconso/rappels.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,categorie_produit,VARCHAR,YES,None,None,None
1,conditionnements,VARCHAR,YES,None,None,None
2,conduites_a_tenir_par_le_consommateur,VARCHAR,YES,None,None,None
3,date_date_fin_commercialisation,TIMESTAMP WITH TIME ZONE,YES,None,None,None
4,date_de_fin_de_la_procedure_de_rappel,TIMESTAMP WITH TIME ZONE,YES,None,None,None
5,date_debut_commercialisation,TIMESTAMP WITH TIME ZONE,YES,None,None,None
6,date_publication,TIMESTAMP WITH TIME ZONE,YES,None,None,None
7,description_complementaire_risque,VARCHAR,YES,None,None,None
8,distributeurs,VARCHAR,YES,None,None,None
9,id,VARCHAR,YES,None,None,None
